# Tutorial 2: <br><br>Deep Echo State Networks for Text Classification

This notebook presents a use case example of the ``EsnTorch`` library. It describes the implementation of a **Deep Echo State Network (Deep ESN)** for text classification on the **TREC-6** dataset (question classification).

The instantiation, training and evaluation of a Deep ESN for text classification is similar to that of a calssical ESN. It is achieved via the following steps:
1. Import libraries and modules
2. Load and prepare data
3. Instantiate the model:
    - specify parameters
    - specify learning algorithm
    - warm up
4. Train
5. Evaluate

## Librairies

In [1]:
# !pip install transformers==4.8.2
# !pip install datasets==1.7.0
# !pip install tqdm

In [2]:
# To enable progress bars in jupyter:

# pip install ipywidgets
# jupyter nbextension enable --py widgetsnbextension
# conda install -c conda-forge nodejs
# jupyter labextension install @jupyter-widgets/jupyterlab-manager

In [3]:
# Comment this if library is installed!
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

In [4]:
from tqdm.notebook import tqdm_notebook
from sklearn.metrics import classification_report

import torch

from datasets import load_dataset, Dataset, concatenate_datasets

from transformers import AutoTokenizer
from transformers.data.data_collator import DataCollatorWithPadding

import esntorch.core.reservoir as res
import esntorch.core.learning_algo as la
import esntorch.core.pooling_strategy as ps
import esntorch.core.esn as esn

In [5]:
%load_ext autoreload
%autoreload 2

In [6]:
# Set device (cpu or gpu if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

## Load and prepare data

### Load and tokenize data

In [7]:
# Custom functions for loading and preparing data
def tokenize(sample):
    """Tokenize sample"""
    
    sample = tokenizer(sample['text'], truncation=True, padding=False, return_length=True)
    
    return sample
    
def load_and_prepare_dataset(dataset_name, split, cache_dir):
    """
    Load dataset from the datasets library of HuggingFace.
    Tokenize and add length.
    """
    
    # Load dataset
    dataset = load_dataset(dataset_name, split=split, cache_dir=CACHE_DIR)
    
    # Rename label column for tokenization purposes (use 'label-fine' for fine-grained labels)
    dataset = dataset.rename_column('label-coarse', 'labels')
    
    # Tokenize data
    dataset = dataset.map(tokenize, batched=True)
    dataset = dataset.rename_column('length', 'lengths')
    dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels', 'lengths'])
    
    return dataset

In [8]:
# Load BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Load and prepare data
CACHE_DIR = 'cache_dir/' # put your path here

full_dataset = load_and_prepare_dataset('trec', split=None, cache_dir=CACHE_DIR)
train_dataset = full_dataset['train'].sort("lengths")
test_dataset = full_dataset['test'].sort("lengths")

# Create dict of all datasets
dataset_d = {
    'train': train_dataset,
    'test': test_dataset
    }

Using the latest cached version of the module from C:\Users\AJYH\.cache\huggingface\modules\datasets_modules\datasets\trec\751da1ab101b8d297a3d6e9c79ee9b0173ff94c4497b75677b59b61d5467a9b9 (last modified on Tue Mar  3 11:36:19 2026) since it couldn't be found locally at trec, or remotely on the Hugging Face Hub.


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [9]:
dataset_d

{'train': Dataset({
     features: ['labels', 'label-fine', 'text', 'input_ids', 'token_type_ids', 'attention_mask', 'lengths'],
     num_rows: 5452
 }),
 'test': Dataset({
     features: ['labels', 'label-fine', 'text', 'input_ids', 'token_type_ids', 'attention_mask', 'lengths'],
     num_rows: 500
 })}

### Create dataloaders

In [10]:
# Create dict of dataloaders

dataloader_d = {}

for k, v in dataset_d.items():
    dataloader_d[k] = torch.utils.data.DataLoader(v, batch_size=256, 
                                                  collate_fn=DataCollatorWithPadding(tokenizer))

In [11]:
dataloader_d

{'train': <torch.utils.data.dataloader.DataLoader at 0x1f1ddc00fd0>,
 'test': <torch.utils.data.dataloader.DataLoader at 0x1f1ddc00940>}

## Instanciate the model

### Parameters

For Deep ESNs, set the parameter ``deep``to ``True``, then customize the number of layers (i.e., reservoirs) by specifying the parameter ``nb_layers``. Each of the other parameters (like ``dim``, ``distribution``, ``spectal radius``, etc.), can be specified in two ways:
- list of values: in this case, the successive layers are built according to the successive values of the parameter in the list;
- single value: in this case, the successive layers are all built according to the same value of the parameter.
Please refer to the documentation for further details.

In [12]:
# Deep ESN parameters
esn_params = {
            'embedding': 'bert-base-uncased',
            'dim': [500, 400, 300],      # *** list of dims for the successive layers ***
            'sparsity': 0.9,
            'spectral_radius': 0.9,
            'leaking_rate': 0.5,
            'activation_function': 'tanh',
            'bias_scaling': 0.1,
            'input_scaling': 0.1,
            'mean': 0.0,
            'std': 1.0,     
            'learning_algo': None,       # initialzed below
            'criterion': None,           # initialzed below (only for learning algos trained with SGD)
            'optimizer': None,           # initialzed below (only for learning algos trained with SGD)
            'pooling_strategy': 'mean',
            'bidirectional': False,      # True, False
            'mode' : 'recurrent_layer',  # 'no_layer', 'linear_layer', 'recurrent_layer'
            'deep' : True,               # *** Deep ESN ***
            'nb_layers' : 3,             # *** 3 layers ***
            'device': device,
            'seed': 42
            }

# Instantiate the ESN
ESN = esn.EchoStateNetwork(**esn_params)

Model downloaded: bert-base-uncased
Model downloaded: bert-base-uncased


### Learning algorithm
Choose your learning algo by un-commenting its associated cell.

The following algorithms are trained via a **direct method**.<br>
Hence, there is no need to specify any *criterion* and *optimizer*.
- ``RidgeRegression`` (our implementation)
- ``RidgeRegression_skl`` (from scikit-learn)
- ``LinearSVC`` (from scikit-learn)
- ``LogisticRegression_skl`` (from scikit-learn)

In [13]:
# ESN.learning_algo = la.RidgeRegression(alpha=10.0)

In [14]:
# ESN.learning_algo = la.RidgeRegression_skl(alpha=10.0)

In [15]:
# ESN.learning_algo = la.LinearSVC(C=1.0)

In [16]:
# ESN.learning_algo = la.LogisticRegression_skl()

The following algorithms are trained via a **gradient descent**.<br>
Accordingly, a *criterion* and an *optimizer* must be specified.
- ``LogisticRegression`` (our implementation)
- ``DeepNN`` (our implementation)

In [17]:
input_dim = sum([layer.dim for layer in ESN.layer.layers])

if esn_params['bidirectional']:
    input_dim *= 2

In [18]:
ESN.learning_algo = la.LogisticRegression(input_dim=input_dim, output_dim=6)

In [19]:
# ESN.learning_algo = la.DeepNN([input_dim, 512, 256, 6])

In [20]:
# Needs criterion and otpimizer

ESN.criterion = torch.nn.CrossEntropyLoss()
ESN.optimizer = torch.optim.Adam(ESN.learning_algo.parameters(), lr=0.01)

### Warm up

In [21]:
# Put model on device
ESN = ESN.to(device)

In [22]:
if isinstance(ESN.layer, res.LayerRecurrent):
    ESN.warm_up(dataset_d['train'].select(range(10)))

## Train

For **direct methods**, the parameters ``epochs`` and ``iter_steps`` are ignored. 

In [23]:
ESN.fit(dataloader_d["train"], epochs=3, iter_steps=50)

Performing gradient descent...


  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/22 [00:00<?, ?it/s]

  0%|          | 0/22 [00:00<?, ?it/s]

  0%|          | 0/22 [00:00<?, ?it/s]

Iteration: 50 Loss: 1.3088613748550415

Training complete.


[1.3088613748550415]

## Evaluate

In [24]:
# Train predictions and accuracy
train_pred, train_acc = ESN.predict(dataloader_d["train"], verbose=False)
train_acc

  0%|          | 0/22 [00:00<?, ?it/s]

59.9413059427733

In [25]:
# Test predictions
test_pred, test_acc = ESN.predict(dataloader_d["test"], verbose=False)
test_acc

  0%|          | 0/2 [00:00<?, ?it/s]

59.2

In [26]:
# Test classification report
print(classification_report(test_pred.tolist(), 
                            dataset_d['test']['labels'].tolist(), 
                            digits=4))

              precision    recall  f1-score   support

           0     0.8986    0.6631    0.7631       187
           1     0.1170    0.9167    0.2075        12
           2     0.2222    1.0000    0.3636         2
           3     0.9846    0.3422    0.5079       187
           4     0.3097    0.9459    0.4667        37
           5     0.7407    0.8000    0.7692        75

    accuracy                         0.5920       500
   macro avg     0.5455    0.7780    0.5130       500
weighted avg     0.8420    0.5920    0.6317       500

